In [45]:
!pip install numpy

In [46]:
import torch
from torch import nn, einsum
import numpy as np
!pip install einops
from einops import rearrange

import torch.nn.functional as f

In [47]:
class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement
    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1,2))

In [48]:
x = torch.linspace(1,81,81).view(9,9)
print(x)
y = torch.roll(input = x, shifts=(-1,-1), dims=(0,1))
print(y)

tensor([[ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18.],
        [19., 20., 21., 22., 23., 24., 25., 26., 27.],
        [28., 29., 30., 31., 32., 33., 34., 35., 36.],
        [37., 38., 39., 40., 41., 42., 43., 44., 45.],
        [46., 47., 48., 49., 50., 51., 52., 53., 54.],
        [55., 56., 57., 58., 59., 60., 61., 62., 63.],
        [64., 65., 66., 67., 68., 69., 70., 71., 72.],
        [73., 74., 75., 76., 77., 78., 79., 80., 81.]])
tensor([[11., 12., 13., 14., 15., 16., 17., 18., 10.],
        [20., 21., 22., 23., 24., 25., 26., 27., 19.],
        [29., 30., 31., 32., 33., 34., 35., 36., 28.],
        [38., 39., 40., 41., 42., 43., 44., 45., 37.],
        [47., 48., 49., 50., 51., 52., 53., 54., 46.],
        [56., 57., 58., 59., 60., 61., 62., 63., 55.],
        [65., 66., 67., 68., 69., 70., 71., 72., 64.],
        [74., 75., 76., 77., 78., 79., 80., 81., 73.],
        [ 2.,  3.,  4.,  5.,  6.,  7.,  8.,  9.,  1.]])


In [49]:
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn =fn
    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) +x

In [50]:
class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kwargs):
        #return self.fn(self.norm(x), **kwargs) # pre norm for Version 1
        return self.norm(self.fn(x), **kwargs)  # post norm for version 2

In [51]:
torch.manual_seed(0)
B,H,W,C =1,2,2,3
input = torch.randn(B,H,W,C)*100
print('input: ', input)
layer_norm = nn.LayerNorm(C)
output = layer_norm(input)
print('output: ',output)

input:  tensor([[[[ 154.0996,  -29.3429, -217.8789],
          [  56.8431, -108.4522, -139.8595]],

         [[  40.3347,   83.8026,  -71.9258],
          [ -40.3344,  -59.6635,   18.2036]]]])
output:  tensor([[[[ 1.2191,  0.0112, -1.2303],
          [ 1.3985, -0.5173, -0.8813]],

         [[ 0.3495,  1.0120, -1.3615],
          [-0.3948, -0.9787,  1.3735]]]], grad_fn=<NativeLayerNormBackward0>)


In [52]:
class FeedForward(nn.Module):
    # mlp_dim = hidden_dim*4 where dim = hidden_dim=(96,192,384,768)
    def __init__(self, dim,hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim,dim),
        )
    def forward(self,x):
        return self.net(x)

In [53]:
def create_mask(window_size, displacement, upper_lower, left_right):  #to handle cycleshifted patches
        mask = torch.zeros(window_size**2, window_size**2) #(49,49)
        print('original mask: ', mask)
        if upper_lower:
            mask[-displacement* window_size:, :-displacement*window_size] = float('-inf') #down left section
            mask[:-displacement*window_size, -displacement*window_size:] =float('-inf') # up right section
        if left_right:
            mask = rearrange(mask, '(h1 w1) (h2 w2)->h1 w1 h2 w2', h1= window_size, h2 = window_size)
            mask[:, -displacement:,:,:-displacement] = float('-inf')
            mask[:,:-displacement, :, -displacement:] = float('-inf')
            mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')
        return mask

In [54]:
def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x,y] for x in range(window_size) for y in range(window_size)]), dtype=torch.long)
    #print('indices: ', indices.size())                         #(49,2)[0,0], [0,1],....,[0,6],....,[6,6]
    dis = indices[None, : , :]- indices[:,None,:]
    return dis

In [55]:
tem = get_relative_distances(window_size=3)
print(tem.size())
print(tem[:,:,0])

torch.Size([9, 9, 2])
tensor([[ 0,  0,  0,  1,  1,  1,  2,  2,  2],
        [ 0,  0,  0,  1,  1,  1,  2,  2,  2],
        [ 0,  0,  0,  1,  1,  1,  2,  2,  2],
        [-1, -1, -1,  0,  0,  0,  1,  1,  1],
        [-1, -1, -1,  0,  0,  0,  1,  1,  1],
        [-1, -1, -1,  0,  0,  0,  1,  1,  1],
        [-2, -2, -2, -1, -1, -1,  0,  0,  0],
        [-2, -2, -2, -1, -1, -1,  0,  0,  0],
        [-2, -2, -2, -1, -1, -1,  0,  0,  0]])


In [56]:
class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        # dim = hidden_dim = (96,192,384,768);
        # heads = num_heads= (3,6,12,24);
        # head_dim=32
        super().__init__()
        inner_dim = head_dim * heads  #(32*3=96, 32*6=192, 32*12=384, 32*24=768) =C
        self.heads = heads
        self.scale = head_dim** -0.5 # scalling dot product inside the softmax
        self.window_size = window_size # window_Size =7 
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted =shifted

        self.tau = nn.Parameter(torch.tensor(0.01), requires_grad=True)
        if self.shifted:
            displacement = window_size //2 #7//2 =3
            self.cyclic_shift = CyclicShift(-displacement)
            self.cyclic_back_shift = CyclicShift(displacement)
            #(49,49): masks are NOT learnable parameters; requires_grad=False;
            self.upper_lower_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                             upper_lower=True, left_right=False), requires_grad=False)
            self.left_right_mask = nn.Parameter(create_mask(window_size=window_size, displacement = displacement,
                                                            upper_lower=False, left_right=True),  requires_grad=False)
        self.to_qkv = nn.Linear(dim, inner_dim*3, bias=False)
        #dim (96,192,384,768) and (inner_dim = head_dim* heads): we can also use C*3 and gives us same thing
        #self.pos_embedding = nn.Parameter(torch.randn(window_size**2, window_size**2)) #(49,49)
        if self.relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size)+ window_size -1
            self.pos_embedding = nn.Parameter(torch.randn(2*window_size-1 , 2*window_size-1))
            #(13,13) because if I am on one cell I have 6 possible relationship behind and after
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size**2, window_size**2)) #(49,49)
        self.to_out = nn.Linear(inner_dim, dim)
        #inner_dim = head_dim* heads=C, dim = hidden_dim = (96,192,384,768)
    def forward(self,x):
        if self.shifted:
            #print('x size: ', x.size())  #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
            x =self.cyclic_shift(x)
            #print('x size: ', x.size()) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
        b, n_h, n_w, _, h = *x.shape, self.heads 
        #print('x.shape: ', x.shape)  #(1,(56,28,14,7), (56,28,14,7), (96,192,384,768))
        #print('self.to_qkv(x): ', self.to_qkv(x).size()) #(1,(56,28,14,7), (56,28,14,7), (288,576,1152,2304))
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        #print('qkv: ', qkv[0].size())
        #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768)) for qkv[0] as qkv is a tuple
        nw_h = n_h // self.window_size  #(56//7=8, 28//7 =4, 14//7=2, 7//7=1)
        nw_w = n_w // self.window_size  #(56//7=8, 28//7 =4, 14//7=2, 7//7=1)

        q,k,v = map(lambda t: rearrange(t, 'b (nw_h w_h) (nw_w w_w)(h d)-> b h(nw_h nw_w) (w_h w_w) d',
                                        h=h, w_h=self.window_size, w_w=self.window_size), qkv)
        #print('q size: ', q.size())
        #(b=1, h=(3,6,12,24), (nw_h*nw_w)=(64,16,4,1), (w_h* w_w)=49, d=32) where d =head_dim; h=#heads;
        #print('k size: ', k.size())        #same as q
        #print('v size: ', v.size())        #same as q

        #Dot product similarity
        #dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale
        # b = batch_size , h =#heads(3,6,12,24), w = (64,16,4,1), i=49, j =49

        #first normalizing q and k with respect to each row
        q=  f.normalize(q,p=2, dim=-1)
        k = f.normalize(k, p=2, dim=-1)
        #cosine similarity
        dots = einsum('b h w i d, b h w j d -> b h w i j', q,k)/self.tau
        #dots += self.pos_embedding
        #b = batch_size, h=#heads (3,6,12,24), w=(64,16,4,1), i=49, j =49
        if self.relative_pos_embedding:
            #print('self.pos_embedding: ', self.pos_embedding) #(13,13)
            tmp1= self.relative_indices[:,:,0]  #(49,49) as relative_indices is (49,49,2)
            #tmp2 = self.pos_embedding[self.relative_indices[:,:,0], self.relative_indices[:,:,1]] #(49,49)
            #print(tmp2.size())
            dots+= self.pos_embedding[self.relative_indices[:,:,0], self.relative_indices[:,:,1]] #b = batch_size, h=#heads(3,6,12,2)
        else:
            dots+=self.pos_embedding
            
        if self.shifted:
            #temp1 = self.upper_lower_mask #(49,49)
            #temp2 = self.left_right_mask #(49,49)
            dots[:, :,-nw_w:] +=self.upper_lower_mask
            dots[:,:, nw_w-1::nw_w] += self.left_right_mask
        attn = dots.softmax(dim=-1)
        #b = batch_size, h= #heads (3,6,12,24), w=(64,16,4,1), i=49, j =49

        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        #(b=1, h=(3,6,12,24), (nw_h * nw_w)= (64,16,,4,1), (w_h*w_w)=49, d=32) where d =head_dim; h=#heads;
        out = rearrange(out, 'b h (nw_h nw_w) (w_h w_w) d-> b (nw_h w_h) (nw_w w_w) (h d)',
                        h = h, w_h= self.window_size, w_w=self.window_size, nw_h= nw_h, nw_w=nw_w)
        #(1,(56,28,14,7), (56,28,14,7), (96,192,384,768))

        out =self.to_out(out) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))

        if self.shifted:
            out = self.cyclic_back_shift(out) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
        return out

In [57]:
p =torch.tensor([[1,2],[3,4]])
print(p.size())
r = torch.tensor([[[0,0],[0,0],[0,0]],
                  [[1,1], [1,1],[1,1]],
                  [[0,1],[0,1],[0,1]]
                  ])
print(r.size())
print(p[r[:,:, 0], r[:,:,1]])

torch.Size([2, 2])
torch.Size([3, 3, 2])
tensor([[1, 1, 1],
        [4, 4, 4],
        [2, 2, 2]])


In [58]:
do = torch.rand(1,1,2,5,5)
print('do: ', do)
po = torch.ones(5,5)
print('po: ', po)
su = do + po
print('do3: ', su)

do[:,:,1] =do[:,:,1]+ po
print('do3: ', do)

do:  tensor([[[[[0.4194, 0.5529, 0.9527, 0.0362, 0.1852],
           [0.3734, 0.3051, 0.9320, 0.1759, 0.2698],
           [0.1507, 0.0317, 0.2081, 0.9298, 0.7231],
           [0.7423, 0.5263, 0.2437, 0.5846, 0.0332],
           [0.1387, 0.2422, 0.8155, 0.7932, 0.2783]],

          [[0.4820, 0.8198, 0.9971, 0.6984, 0.5675],
           [0.8352, 0.2056, 0.5932, 0.1123, 0.1535],
           [0.2417, 0.7262, 0.7011, 0.2038, 0.6511],
           [0.7745, 0.4369, 0.5191, 0.6159, 0.8102],
           [0.9801, 0.1147, 0.3168, 0.6965, 0.9143]]]]])
po:  tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
do3:  tensor([[[[[1.4194, 1.5529, 1.9527, 1.0362, 1.1852],
           [1.3734, 1.3051, 1.9320, 1.1759, 1.2698],
           [1.1507, 1.0317, 1.2081, 1.9298, 1.7231],
           [1.7423, 1.5263, 1.2437, 1.5846, 1.0332],
           [1.1387, 1.2422, 1.8155, 1.7932, 1.2783]],

          [[1.4820, 1.8198, 1.

In [59]:
create_mask(window_size=3, displacement=1, upper_lower=True, left_right =False)

original mask:  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.]])


tensor([[0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf, -inf, -inf, 0., 0., 0.],
        [-inf, -inf, -inf, -inf, -inf, -inf, 0., 0., 0.],
        [-inf, -inf, -inf, -inf, -inf, -inf, 0., 0., 0.]])

In [60]:
class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        # dim = hidden_Dim=(96,192,384,768): #heads = num_heads = (3,6,12,24); mlp_dim = hidden_dim*4
        super().__init__()
        self.attention_block = Residual(PreNorm(dim, WindowAttention(
                                                                dim =dim,
                                                                heads=heads,
                                                                head_dim =head_dim,
                                                                shifted =shifted,
                                                                window_size =window_size,
                                                                relative_pos_embedding=relative_pos_embedding
        )))
        self.mlp_block = Residual(PreNorm(dim, FeedForward(dim=dim, hidden_dim= mlp_dim)))
    def forward(self,x):
        x = self.attention_block(x) # (1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
        x =self. mlp_block(x) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
        return x

In [61]:
class PatchMerging_Conv(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.patch_merge = nn.Conv2d(in_channels,
                                     out_channels,
                                     kernel_size=downscaling_factor, 
                                     stride=downscaling_factor,
                                     padding=0)
    def forward(self, x):
        #print('x.size: ', x.size()) #(1,(3,96,192,384), (224, 56,28,14), (224, 56, 28,14))
        #self.patch_merge(x). #(1,(96,192,384,768), (56,28,14,7), (56,28,14,7))
        x = self.patch_merge(x).permute(0,2,3,1) #(1,(56,28,14,7), (56,28,14,7), (96,192,384,768))
        return x

In [62]:
class StageModule(nn.Module):
    def __init__(self, in_channels, hidden_dimension, layers, downscaling_factor, num_heads, head_dim, window_size,
                 relative_pos_embedding):
        super().__init__()
        assert layers %2 ==0, 'Stage layers need to be divisible by 2 for regular and shifted block.'
        self.patch_partition = PatchMerging_Conv(in_channels=in_channels, out_channels = hidden_dimension,
                                                 downscaling_factor=downscaling_factor)
        self.layers = nn.ModuleList([])
        for _ in range(layers //2):
            self.layers.append(nn.ModuleList([
                SwinBlock(dim= hidden_dimension, heads = num_heads, head_dim=head_dim, mlp_dim = hidden_dimension*4,
                          shifted=False, window_size= window_size, relative_pos_embedding=relative_pos_embedding),
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim = hidden_dimension*4,
                          shifted=True, window_size= window_size, relative_pos_embedding= relative_pos_embedding),
            ]))
    def forward(self,x):
        # print('before patch merging: ', x.size()) 
        x=self.patch_partition(x) # (1,(3,96,192,384), (224,56,28,14), (224,56,28,14))
        #print( 'after path merging: ', x.size())  # (1,(56,28,14,7), (56,28,14,7), (96,192,384,768))
        for regular_block, shifted_block in self.layers:
            x = regular_block(x) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
            x = shifted_block(x) #(1, (56,28,14,7), (56,28,14,7), (96,192,384,768))
        return x.permute(0,3,1,2)

In [63]:
class SwinTransformer(nn.Module):
    def __init__(self, hidden_dim, layers, heads, channels=3, num_classes=1000, head_dim=32, window_size=7, 
                    downscaling_factors=(4,2,2,2),  relative_pos_embedding=True       ):
        super().__init__()

        self.stage1= StageModule(in_channels= channels, hidden_dimension = hidden_dim, layers= layers[0],
                                 downscaling_factor=downscaling_factors[0], num_heads = heads[0], head_dim= head_dim,
                                 window_size=window_size, relative_pos_embedding = relative_pos_embedding)
        self.stage2= StageModule(in_channels= hidden_dim, hidden_dimension = hidden_dim*2, layers= layers[1],
                                 downscaling_factor=downscaling_factors[1], num_heads = heads[1], head_dim= head_dim,
                                 window_size=window_size, relative_pos_embedding = relative_pos_embedding)
        self.stage3= StageModule(in_channels= hidden_dim*2, hidden_dimension = hidden_dim*4, layers= layers[2],
                                 downscaling_factor=downscaling_factors[2], num_heads = heads[2], head_dim= head_dim,
                                 window_size=window_size, relative_pos_embedding = relative_pos_embedding)
        self.stage4= StageModule(in_channels= hidden_dim*4, hidden_dimension = hidden_dim*8, layers= layers[3],
                                 downscaling_factor=downscaling_factors[3], num_heads = heads[3], head_dim= head_dim,
                                 window_size=window_size, relative_pos_embedding = relative_pos_embedding)
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(hidden_dim*8),
            nn.Linear(hidden_dim*8, num_classes)
        )
    def forward(self, img):
        x =self.stage1(img)
        x =self.stage2(x)
        x =self.stage3(x)
        x =self.stage4(x)   #(batch, 768,7,7)
        x =x.mean(dim=[2,3]) #(batch, 1,768)
        return self.mlp_head(x)

In [64]:
def swin_t(hidden_dim=96, layers=(2,2,6,2), heads=(3,6,12,24), **kwargs):
    return SwinTransformer(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)

In [65]:
net = swin_t(
    hidden_dim=96,
    layers=(2,2,6,2),
    heads = (3,6,12,24),
    channels =3,
    num_classes =3,
    head_dim =32,
    window_size=7,
    downscaling_factors = (4,2,2,2),
    relative_pos_embedding =True
)
dummy_x =torch.randn(1,3,224,224)
logits = net(dummy_x) #(1,3)
print(net)


original mask:  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
original mask:  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
original mask:  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
original mask:  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
 

In [66]:
print(logits)

tensor([[-0.3987, -0.3263,  1.0310]], grad_fn=<AddmmBackward0>)
